# Online SAC with GeneticBatch Fallback

This notebook loads the completed eight-dataset online run, summarizes fallback use, and plots one selected dataset without mixing repeated simulation timelines.

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv
import os
import subprocess
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-vec-cache')

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'online_hybrid.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root.')


PROJECT_ROOT = find_project_root()
PRETRAINED_MODEL = PROJECT_ROOT / 'outputs' / 'models' / 'discrete_sac' / 'sac_discrete_best.pt'
ADAPTED_MODEL = PROJECT_ROOT / 'outputs' / 'models' / 'discrete_sac_online' / 'sac_adapted_final.pt'
FULL_RESULTS_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'test_hybrid_results.csv'
FULL_SWITCH_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'test_hybrid_switches.csv'
NOTEBOOK_RESULTS_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'notebook_hybrid_results.csv'
NOTEBOOK_SWITCH_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'notebook_hybrid_switches.csv'

print('Project:', PROJECT_ROOT)
print('Pretrained model:', PRETRAINED_MODEL)
print('Adapted model:', ADAPTED_MODEL)

## Optional Short Run

Leave `RUN_EVALUATION = False` to inspect the complete run you already generated. The optional run writes separate notebook CSV files and does not replace the full results.

In [ ]:
RUN_EVALUATION = False
ENABLE_FALLBACK_FINETUNING = True
RUN_DATASET = 'test_slow_mix'
MAX_TIMESTEPS = 400

WINDOW_SIZE = 100
CHECK_EVERY = 20
FALLBACK_LOSS_RATE = 0.05
RECOVERY_LOSS_RATE = 0.02
FALLBACK_MISS_RATE = 0.10
RECOVERY_MISS_RATE = 0.03
FALLBACK_LATENCY = 1.00
RECOVERY_LATENCY = 0.70

if RUN_EVALUATION:
    command = [
        sys.executable, str(PROJECT_ROOT / 'src' / 'online_hybrid.py'),
        '--split', 'test',
        '--dataset', RUN_DATASET,
        '--max-timesteps', str(MAX_TIMESTEPS),
        '--checkpoint', str(PRETRAINED_MODEL),
        '--adapted-model', str(PROJECT_ROOT / 'outputs' / 'models' / 'discrete_sac_online' / 'notebook_adapted.pt'),
        '--results-file', str(NOTEBOOK_RESULTS_FILE),
        '--switch-file', str(NOTEBOOK_SWITCH_FILE),
        '--window-size', str(WINDOW_SIZE),
        '--check-every', str(CHECK_EVERY),
        '--fallback-loss-rate', str(FALLBACK_LOSS_RATE),
        '--recovery-loss-rate', str(RECOVERY_LOSS_RATE),
        '--fallback-miss-rate', str(FALLBACK_MISS_RATE),
        '--recovery-miss-rate', str(RECOVERY_MISS_RATE),
        '--fallback-latency', str(FALLBACK_LATENCY),
        '--recovery-latency', str(RECOVERY_LATENCY),
    ]
    if not ENABLE_FALLBACK_FINETUNING:
        command.append('--no-train')
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
    subprocess.run(command, cwd=PROJECT_ROOT, env=environment, check=True)

RESULTS_FILE = NOTEBOOK_RESULTS_FILE if RUN_EVALUATION else FULL_RESULTS_FILE
SWITCH_FILE = NOTEBOOK_SWITCH_FILE if RUN_EVALUATION else FULL_SWITCH_FILE
if not RESULTS_FILE.exists():
    raise FileNotFoundError(f'Missing online results: {RESULTS_FILE}')

## Overall Hybrid Summary

In [ ]:
with RESULTS_FILE.open(newline='', encoding='utf-8') as handle:
    all_rows = list(csv.DictReader(handle))

rows_by_dataset = defaultdict(list)
for row in all_rows:
    rows_by_dataset[row['dataset']].append(row)

print('Results:', RESULTS_FILE)
print('Switch log:', SWITCH_FILE)
for dataset, dataset_rows in rows_by_dataset.items():
    count = len(dataset_rows)
    ga_tasks = sum(row['controller_source'] == 'GA' for row in dataset_rows)
    switches = sum(bool(row['switch_event']) for row in dataset_rows)
    misses = sum(row['deadline_missed'].lower() == 'true' for row in dataset_rows)
    losses = sum(row['packet_lost'].lower() == 'true' for row in dataset_rows)
    latency = sum(float(row['latency']) for row in dataset_rows) / count
    energy = sum(float(row['total_system_energy']) for row in dataset_rows) / count
    print(
        f'{dataset:20s} tasks={count:6d} GA={100 * ga_tasks / count:6.1f}% '
        f'switches={switches:3d} miss={misses:5d} loss={losses:4d} '
        f'latency={latency:8.3f} energy={energy:7.3f}'
    )

## Inspect One Dataset

Change `SELECTED_DATASET` to any name printed above, then rerun this cell and the plot cell.

In [ ]:
SELECTED_DATASET = 'test_slow_mix'
if SELECTED_DATASET not in rows_by_dataset:
    raise ValueError(f'Unknown dataset. Choose from: {list(rows_by_dataset)}')

rows = [dict(row) for row in rows_by_dataset[SELECTED_DATASET]]
for row in rows:
    for field in ('release_time', 'reward', 'latency', 'normalized_latency', 'total_system_energy'):
        row[field] = float(row[field])
    for field in ('rolling_loss_rate', 'rolling_deadline_miss_rate', 'rolling_normalized_latency'):
        row[field] = float(row[field]) if row[field] else np.nan
    for field in ('transmission_loss_event', 'packet_lost', 'deadline_missed'):
        row[field] = row[field].lower() == 'true'

times = np.array([row['release_time'] for row in rows])
weather_code = {'BASE': 0, 'RAIN': 1, 'SNOW': 2, 'FOG': 3}
weather = np.array([weather_code[row['scenario']] for row in rows])
ga_active = np.array([row['controller_source'] == 'GA' for row in rows], dtype=float)
rolling_loss = np.array([row['rolling_loss_rate'] for row in rows])
rolling_miss = np.array([row['rolling_deadline_miss_rate'] for row in rows])
rolling_latency = np.array([row['rolling_normalized_latency'] for row in rows])
reward = np.array([row['reward'] for row in rows])
latency = np.array([row['latency'] for row in rows])
energy = np.array([row['total_system_energy'] for row in rows])
packet_lost = np.array([row['packet_lost'] for row in rows], dtype=int)
deadline_missed = np.array([row['deadline_missed'] for row in rows], dtype=int)
switches = [row for row in rows if row['switch_event']]

print('Selected:', SELECTED_DATASET)
print('Tasks:', len(rows), 'Switches:', len(switches))

In [ ]:
def rolling_mean(values, window=100):
    values = np.asarray(values, dtype=float)
    result = np.full(len(values), np.nan)
    if len(values) >= window:
        result[window - 1:] = np.convolve(values, np.ones(window) / window, mode='valid')
    return result


figure, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
figure.suptitle(f'Online hybrid controller: {SELECTED_DATASET}', fontsize=15)

axes[0, 0].step(times, weather, where='post', color='#287271')
axes[0, 0].set_yticks(list(weather_code.values()), list(weather_code.keys()))
axes[0, 0].set_title('Weather stream')

axes[0, 1].fill_between(times, 0, ga_active, step='post', color='#E07A5F', alpha=0.8)
axes[0, 1].set_yticks([0, 1], ['SAC', 'GA'])
axes[0, 1].set_title('Controller source')

axes[1, 0].plot(times, rolling_loss, color='#C44536', label='loss-event rate')
axes[1, 0].plot(times, rolling_miss, color='#6A4C93', label='deadline-miss rate')
axes[1, 0].plot(times, rolling_latency, color='#287271', label='normalized latency')
axes[1, 0].axhline(FALLBACK_LOSS_RATE, color='#C44536', linestyle=':', alpha=0.7)
axes[1, 0].axhline(FALLBACK_MISS_RATE, color='#6A4C93', linestyle=':', alpha=0.7)
axes[1, 0].axhline(FALLBACK_LATENCY, color='#287271', linestyle=':', alpha=0.7)
axes[1, 0].set_title('Hybrid monitor signals')
axes[1, 0].legend()

axes[1, 1].plot(times, rolling_mean(reward), color='#264653')
axes[1, 1].set_title('Rolling mean reward')

axes[2, 0].plot(times, rolling_mean(latency), color='#3A86FF', label='latency')
energy_axis = axes[2, 0].twinx()
energy_axis.plot(times, rolling_mean(energy), color='#F4A261', label='energy')
axes[2, 0].set_ylabel('latency (s)')
energy_axis.set_ylabel('total system energy')
axes[2, 0].set_title('Rolling latency and energy')

axes[2, 1].plot(times, np.cumsum(packet_lost), color='#D62828', label='packet losses')
axes[2, 1].plot(times, np.cumsum(deadline_missed), color='#6A4C93', label='deadline misses')
axes[2, 1].set_title('Cumulative failures')
axes[2, 1].legend()

for axis in axes.flat:
    for row in switches:
        axis.axvline(row['release_time'], color='black', alpha=0.22, linewidth=1)
    axis.grid(alpha=0.2)
    axis.set_xlabel('simulation time (s)')

plt.tight_layout()
plt.show()

The vertical lines mark controller switches. Recovery is measured from GA-executed outcomes, so a later return to GA means SAC had not recovered sufficiently after regaining control.